# Eksperimen Sistem Machine Learning - Credit Risk Prediction

Notebook ini digunakan khusus untuk Kriteria 1 Dicoding MSML: eksperimen dataset, EDA, dan preprocessing dataset Credit Risk. Output preprocessing notebook ini disamakan dengan `automate_Vicky-Ferdinand-Samuel.py` dan dataset yang digunakan pada tahap modelling.

## 1. Data Loading

Dataset mentah dibaca dari `../dataset_raw/credit_risk_dataset.csv.zip`. Target prediksi adalah `loan_status` dengan kelas biner 0 dan 1.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.labelsize": 11})

RAW_PATH = Path("../dataset_raw/credit_risk_dataset.csv.zip")
OUTPUT_DIR = Path("dataset_preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {RAW_PATH}")

df = pd.read_csv(RAW_PATH)
print("Shape dataset:", df.shape)
df.head()

## 2. Dataset Overview

Bagian ini memeriksa struktur data, tipe kolom, statistik deskriptif, dan distribusi target. Untuk credit risk, target imbalance perlu diperhatikan karena model dapat terlihat akurat walau gagal menangkap peminjam berisiko.

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
target_counts = df["loan_status"].value_counts().sort_index()
target_pct = df["loan_status"].value_counts(normalize=True).sort_index() * 100
summary_target = pd.DataFrame({"count": target_counts, "percentage": target_pct.round(2)})
print(summary_target)

ax = sns.countplot(data=df, x="loan_status", hue="loan_status", palette="Set2", legend=False)
ax.set_title("Distribusi Target loan_status")
ax.set_xlabel("loan_status")
ax.set_ylabel("Jumlah Data")
plt.show()

**Insight bisnis**: mayoritas pinjaman berada pada kelas `0`, sehingga proses evaluasi model pada tahap berikutnya perlu memperhatikan metrik seperti recall, F1, ROC-AUC, dan average precision, bukan hanya accuracy.

## 3. Missing Value Analysis

Missing value dianalisis sebelum preprocessing agar strategi imputasi dapat dijelaskan. Pada dataset ini missing value terdapat pada lama bekerja dan suku bunga pinjaman.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing_count, "missing_percentage": missing_pct})
missing_df[missing_df["missing_count"] > 0]

**Insight bisnis**: `person_emp_length` dan `loan_int_rate` adalah fitur yang relevan untuk risiko kredit. Karena keduanya numerik dan distribusinya memiliki outlier, median imputation lebih stabil daripada mean imputation.

## 4. Duplicate Analysis

Duplikasi baris dicek karena data duplikat dapat membuat distribusi data bias dan membuat performa model terlihat lebih baik dari kondisi nyata.

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Jumlah baris duplikat:", duplicate_count)

**Insight bisnis**: duplikasi dapat membuat profil peminjam tertentu dihitung berulang. Baris duplikat dihapus sebelum split agar data train/test lebih representatif.

## 5. Outlier Analysis

Outlier dianalisis pada fitur numerik. Dataset ini memiliki nilai logis yang ekstrem, misalnya `person_age` sangat tinggi dan `person_emp_length` yang tidak realistis dibanding umur peminjam.

In [ ]:
numeric_features = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for idx, col in enumerate(numeric_features):
    sns.boxplot(data=df, x=col, ax=axes[idx], color="#7FB3D5")
    axes[idx].set_title(f"Boxplot {col}")
for idx in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[idx])
plt.tight_layout()
plt.show()

logical_outlier_mask = (
    (~df["person_age"].between(18, 100))
    | ((df["person_emp_length"].notna()) & (~df["person_emp_length"].between(0, 80)))
    | ((df["person_emp_length"].notna()) & (df["person_emp_length"] > (df["person_age"] - 14)))
)
print("Jumlah outlier logis:", int(logical_outlier_mask.sum()))
df.loc[logical_outlier_mask, ["person_age", "person_emp_length"]].head()

**Insight bisnis**: umur dan lama bekerja yang tidak masuk akal dapat merusak pola risiko. Baris outlier logis dihapus, sedangkan outlier numerik lain ditangani dengan winsorization 1% dan 99% yang dipelajari dari train set.

## 6. EDA Fitur Numerik dan Kategorikal

EDA digunakan untuk melihat distribusi fitur, segmen pinjaman, dan pola default berdasarkan atribut peminjam.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for idx, col in enumerate(numeric_features):
    sns.histplot(df[col].dropna(), kde=True, bins=30, ax=axes[idx], color="#2F6B7C")
    axes[idx].set_title(f"Distribusi {col}")
for idx in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[idx])
plt.tight_layout()
plt.show()

In [ ]:
categorical_features = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for idx, col in enumerate(categorical_features):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=axes[idx], color="#69A297")
    axes[idx].set_title(f"Distribusi {col}")
plt.tight_layout()
plt.show()

In [ ]:
default_by_grade = df.groupby("loan_grade")["loan_status"].mean().sort_index()
default_by_home = df.groupby("person_home_ownership")["loan_status"].mean().sort_values(ascending=False)
print("Default rate by loan_grade:")
print((default_by_grade * 100).round(2))
print("
Default rate by home ownership:")
print((default_by_home * 100).round(2))

**Insight bisnis**: `loan_grade`, `loan_percent_income`, dan kepemilikan rumah memberi sinyal risiko. Grade pinjaman yang lebih buruk dan rasio pinjaman terhadap pendapatan yang tinggi cenderung berkaitan dengan risiko default yang lebih tinggi.

## 7. Correlation Analysis

Korelasi fitur numerik dengan target membantu mengidentifikasi sinyal awal sebelum modelling.

In [ ]:
corr_cols = numeric_features + ["loan_status"]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap Fitur Numerik")
plt.tight_layout()
plt.show()

**Insight bisnis**: korelasi linear tidak selalu menangkap semua pola risiko kredit, tetapi `loan_percent_income` dan fitur terkait durasi/kredit memberi indikasi awal yang berguna untuk tahap modelling.

## 8. Preprocessing: Split, Missing Value, Outlier Handling, Encoding, Scaling

Pipeline berikut disusun agar identik dengan `automate_Vicky-Ferdinand-Samuel.py` dan dataset yang digunakan pada modelling.

In [ ]:
TARGET = "loan_status"
RANDOM_STATE = 42
NUMERIC_COLS = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"
]
SCALE_COLS = NUMERIC_COLS + ["loan_grade"]
NOMINAL_COLS = ["person_home_ownership", "loan_intent"]
GRADE_MAP = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
DEFAULT_MAP = {"N": 0, "Y": 1}

# Remove duplicate rows.
df_clean = df.drop_duplicates().copy()

# Remove business-rule outliers.
outlier_mask = (
    (df_clean["person_age"].between(18, 100))
    & ((df_clean["person_emp_length"].isna()) | (df_clean["person_emp_length"].between(0, 80)))
    & ((df_clean["person_emp_length"].isna()) | (df_clean["person_emp_length"] <= (df_clean["person_age"] - 14)))
)
df_clean = df_clean.loc[outlier_mask].copy()

train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    stratify=df_clean[TARGET],
    random_state=RANDOM_STATE,
)
train_df = train_df.copy()
test_df = test_df.copy()

medians = train_df[NUMERIC_COLS].median(numeric_only=True)
q01 = train_df[NUMERIC_COLS].quantile(0.01)
q99 = train_df[NUMERIC_COLS].quantile(0.99)

for col in NUMERIC_COLS:
    train_df[col] = train_df[col].fillna(medians[col]).clip(q01[col], q99[col])
    test_df[col] = test_df[col].fillna(medians[col]).clip(q01[col], q99[col])

for data in (train_df, test_df):
    data["loan_grade"] = data["loan_grade"].map(GRADE_MAP).astype(int)
    data["cb_person_default_on_file"] = data["cb_person_default_on_file"].map(DEFAULT_MAP).astype(int)

train_df = pd.get_dummies(train_df, columns=NOMINAL_COLS, drop_first=False, dtype=int)
test_df = pd.get_dummies(test_df, columns=NOMINAL_COLS, drop_first=False, dtype=int)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

scaler = StandardScaler()
train_df[SCALE_COLS] = scaler.fit_transform(train_df[SCALE_COLS])
test_df[SCALE_COLS] = scaler.transform(test_df[SCALE_COLS])

ordered_columns = [col for col in train_df.columns if col != TARGET] + [TARGET]
train_df = train_df[ordered_columns]
test_df = test_df[ordered_columns]
processed_df = pd.concat([train_df, test_df], axis=0)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Processed shape:", processed_df.shape)

## 9. Export Dataset

Dataset hasil preprocessing disimpan ke `preprocessing/dataset_preprocessing` untuk digunakan oleh tahap modelling berikutnya.

In [ ]:
train_path = OUTPUT_DIR / "credit_risk_train.csv"
test_path = OUTPUT_DIR / "credit_risk_test.csv"
processed_path = OUTPUT_DIR / "credit_risk_processed.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
processed_df.to_csv(processed_path, index=False)

print("Dataset preprocessing berhasil disimpan:")
print(train_path)
print(test_path)
print(processed_path)

## 10. Kesimpulan Eksperimen

Preprocessing akhir mencakup penghapusan duplikasi, penghapusan outlier logis, median imputation, winsorization berbasis train set, ordinal/binary/one-hot encoding, StandardScaler, stratified train-test split, dan export dataset. Strategi ini menjaga konsistensi antara eksperimen, automation script, dan data modelling.